# Pancreatic cancer recurrence pipeline on Colab

This notebook clones the repository to Colab's local disk, installs its pinned environment, keeps data/results/checkpoints in Google Drive, and creates a resumable SPECTRE image-embedding run.

Before running: choose **Runtime → Change runtime type → GPU**. The notebook creates `MyDrive/pc-recurrence-prediction/images` and `MyDrive/pc-recurrence-prediction/table`. Upload the *contents* of the curated `dicom_selected` folder into `images/`, and upload the one Excel workbook into `table/`. Patient data stays out of Git, but you should still confirm that using Colab complies with your data-handling requirements.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Edit these if you use a fork, tag/commit, or different Drive folder.
from pathlib import Path

REPO_URL = "https://github.com/sezeriper/pc-recurrence-prediction.git"
REPO_REF = "master"
LOCAL_REPO = Path("/content/pc-recurrence-prediction")
DRIVE_PROJECT = Path("/content/drive/MyDrive/pc-recurrence-prediction")
IMAGES_DIR = DRIVE_PROJECT / "images"
TABLE_DIR = DRIVE_PROJECT / "table"
SPECTRE_RUN = DRIVE_PROJECT / "outputs/image_embeddings/spectre-colab"
for directory in (IMAGES_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Images folder: {IMAGES_DIR}")
print(f"Table folder:  {TABLE_DIR}")

In [ ]:
# Validate Drive inputs. If either is missing, upload it in Google Drive, then
# return here and press Enter. images/ must contain the *contents* of the
# curated dicom_selected directory, not another enclosing dicom_selected folder.
def data_status():
    workbooks = sorted(TABLE_DIR.rglob("*.xlsx"))
    image_files = [
        path
        for path in IMAGES_DIR.rglob("*")
        if path.is_file() and path.name != "curation_manifest.json"
    ]
    manifest = IMAGES_DIR / "curation_manifest.json"
    return workbooks, image_files, manifest


workbooks, image_files, manifest = data_status()
if len(workbooks) != 1 or not image_files or not manifest.is_file():
    print("Data upload required:")
    if not image_files:
        print(f"  1. Upload the contents of curated dicom_selected/ to {IMAGES_DIR}")
    if not manifest.is_file():
        print(f"  2. Copy curation_manifest.json to {IMAGES_DIR}.")
    if len(workbooks) != 1:
        print(f"  3. Upload exactly one .xlsx workbook to {TABLE_DIR} (found {len(workbooks)}).")
    input("After the upload finishes in Google Drive, press Enter to recheck. ")
    workbooks, image_files, manifest = data_status()

if len(workbooks) != 1 or not image_files or not manifest.is_file():
    raise FileNotFoundError(
        f"Expected one .xlsx file in {TABLE_DIR}, DICOM files in {IMAGES_DIR}, and {manifest}."
    )
WORKBOOK = workbooks[0]
CURATED_DICOM = IMAGES_DIR
print(f"Using workbook: {WORKBOOK}")
print(f"Found {len(image_files)} image file(s) in: {CURATED_DICOM}")

## Clone and install
The URL must be readable without credentials. For a private repository, use your preferred GitHub authentication method and never paste a token into a shared notebook or commit it.

In [ ]:
if (LOCAL_REPO / ".git").is_dir():
    print(f"Updating existing checkout: {LOCAL_REPO}")
else:
    if LOCAL_REPO.exists():
        raise RuntimeError(f"{LOCAL_REPO} exists but is not a Git checkout.")
    !git clone "{REPO_URL}" "{LOCAL_REPO}"

# These are intentionally direct shell commands, so each setup step is visible.
!git -C "{LOCAL_REPO}" fetch --all --tags --prune
!git -C "{LOCAL_REPO}" checkout "{REPO_REF}"
if REPO_REF == "master":
    !git -C "{LOCAL_REPO}" pull --ff-only origin master
# Install uv through the active Colab interpreter. The project itself uses Python 3.12.
!python -m pip install --quiet uv
!uv python install 3.12
%cd {LOCAL_REPO}
!uv sync --python 3.12 --extra imaging

In [ ]:
# Use fast local disk for code, but Drive for everything expensive or persistent.
def link_to_drive(name, target):
    local = LOCAL_REPO / name
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink() and local.resolve() == target.resolve():
        return
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        raise RuntimeError(f"{local} exists and is not a symlink.")
    local.symlink_to(target, target_is_directory=True)


for name in ("outputs", ".cache"):
    link_to_drive(name, DRIVE_PROJECT / name)
# Print the curated folder layout. Missing or invalid patient folders are logged
# as skipped by pc-image-embed unless you explicitly add --require-all.
%cd {LOCAL_REPO}
!find "{CURATED_DICOM}" -mindepth 1 -maxdepth 1 -type d -printf "%f\n" | sort
!uv run python -c 'import torch; print("PyTorch:", torch.__version__)'
!nvidia-smi -L || echo "No GPU visible"

## Input data layout
`images/` must directly contain the selected DICOM patient folders — the contents that were inside `outputs/dicom_selected/` after preprocessing. `table/` must contain exactly one `.xlsx` workbook. This notebook does not run the DICOM selection/review workflow.

In [ ]:
# Example expected Drive layout:
# pc-recurrence-prediction/images/<patient-folder>/<selected-dicom-files>
# pc-recurrence-prediction/table/pankreas adeno ca 10 hasta.xlsx

## Generate SPECTRE embeddings (GPU)
The stable Drive-backed run directory plus `--resume` allows a compatible interrupted run to continue. The first run downloads pinned weights to the persistent cache. `pc-image-embed` prints its checkpoint, device, patient-by-patient, cache, skip, and completion progress below. SPECTRE weights are CC-BY-NC-SA and restricted to non-commercial use.

In [ ]:
%cd {LOCAL_REPO}
!uv run pc-image-embed run --encoder spectre \
    --dicom-root "{CURATED_DICOM}" \
    --workbook "{WORKBOOK}" \
    --run-dir "{SPECTRE_RUN}" --resume
print("SPECTRE embedding artifacts:", SPECTRE_RUN)

## What this produces
The run directory contains `image_embeddings.npz`, `patch_embeddings.npz`, `embedding_summary.csv`, and `run_manifest.json`. The current `pc-recurrence-classify train` CLI requires both Merlin and SPECTRE embedding runs, so recurrence-head training is deliberately not included in this SPECTRE-only notebook.

## Optional repository verification
Uncomment these commands if you modify code in Colab.

In [ ]:
# %cd {LOCAL_REPO}
# !uv sync --python 3.12 --extra imaging --group dev
# !uv run ruff check .
# !uv run pytest